In [9]:
from dotenv import load_dotenv, find_dotenv
from pathlib import Path

# Load closest .env (works whether running from repo root or dev/)
load_dotenv(find_dotenv(), override=False)

True

In [10]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
  model="gpt-4o",
  # reasoning_effort="low",
  # verbosity="low"
)

In [ ]:
output_form = """
{
  "isRelated": "(연관이 됐다면 y, 아니라면 n)"
  "reason": "그렇게 판단한 이유"
}
"""

system_prompt = """
당신은 탁월한 분석가 입니다.
입력으로 키워드와 상품의 정보가 주어진 후, 두 데이터가 서로 관련성이 있는지 없는지를 판단합니다.

markdown 문법을 사용하지 마십시오. 출력은 순수한 JSON으로 이루어져야 합니다.
JSON 형식을 제외한 다른 출력은 절대 금지합니다.
양식 이외의 추가적인 정보를 기입하지 마십시오.
판단의 이유는 최대한 간단하게 작성합니다. 최대 150자를 넘기지 마시오.

출력 양식은 다음과 같습니다:

"""


In [12]:
def input_form(keyword: str, product: dict) -> str:
   return f"""
키워드: {keyword}

상품 정보:
  상품명: {product.get("title", "")}
  가격: {product.get("price", "")}
"""

In [13]:
def filter_links(product: dict) -> dict:
  return {
    "title": product.get("title", "잘못된 제목!"),
    "price": product.get("displayed_price", 
                         product.get("original_price", 
                                     product.get("price", "잘못된 가격!")))
  }

In [19]:
from langchain.schema import SystemMessage, HumanMessage

def call_llm(keyword: str, product: dict):

  filtered = filter_links(product=product)
  
  messages = [
      SystemMessage(content=system_prompt + output_form),
      HumanMessage(content=input_form(keyword=keyword, product=filtered)),
  ]

  result = llm.invoke(messages)

  # 결과값의 .content: 출력물
  return result.content.strip()

In [ ]:
example_keyword = "패딩"

example_product = {
  "title": "ROKA후리스 인기템모음전 인싸템후리스",
  "original_price": "36,000원",
  "displayed_price": "19,800원",
  "product_link": "https://ader.naver.com/v1/U4vv6ASE4bTqy0QU8NWqQi7iYSyB2wleHn55I78MgFQxda2arqoE8WyM-_tt8bV5eUHu41OfsdReRzzaPZuQH0UWJdrOhNuUweeEMTqZfC7EKle5KVtGkTuCF--t1g9-cgp3PxdGdYDIuMH5GHCUQuavqrcRpnMH34B50D5w1GpKM01ZUj48enUN1q1a2UPPpRcW3N91OxIhaY3vETg2h87j3uhMT7OuWyXgh1r8gN-Luoe3IdSXZJfdWPKPdlg287x7-CIRwf8wyLYR6eyUGMJ4YcSnsWzS64mNiZc4ZNButpwXhr3RQ7s3qDEjmO4tPKvmrlRtdSi-nGm1Ss32Ha86vgxHcPpbcPUPpArgXP3PdxWeoV37hjFwuqkM9Iy8KPDly6hzpEpFPI8FR4njbY2_KOok8VHwtGlzCHLdH-n_x2elN3_Ng3FzuHk94OsEOmn6PhlelMmAXFdxEoSZheQUbHovhVThnnvLP8L4NkymhzjDihthDmw9_QNBIozG2O5mwWDcjYowHX3krNhEScwJYeuUN1KwoCM_EPQTpj5YJD1NytS-5CQ6HOs-ESuFSI2u7_jJVaqJIsuehOuG6VlRC4RRoxJ0wV-uS4Z4_jtqcAEABpeFErXVMc6L6AIJ?c=pc.nplusstore.npla&t=0",
  "thumbnail_url": "https://shopping-phinf.pstatic.net/main_8140825/81408254132.5.jpg?type=f300"
}

In [21]:
result = call_llm(example_keyword, example_product)

In [23]:
print(result)

{
  "isRelated": "n",
  "reason": "키워드 '패딩'과 상품명에 포함된 '후리스'는 다른 종류의 의류로, 관련성이 없어 보입니다."
}
